In [16]:
import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Rename columns
train_df.columns = ["label", "title", "text"]
test_df.columns = ["label", "title", "text"]

print(train_df.head())

   label                                              title  \
0      3  Wall St. Bears Claw Back Into the Black (Reuters)   
1      3  Carlyle Looks Toward Commercial Aerospace (Reu...   
2      3    Oil and Economy Cloud Stocks' Outlook (Reuters)   
3      3  Iraq Halts Oil Exports from Main Southern Pipe...   
4      3  Oil prices soar to all-time record, posing new...   

                                                text  
0  Reuters - Short-sellers, Wall Street's dwindli...  
1  Reuters - Private investment firm Carlyle Grou...  
2  Reuters - Soaring crude prices plus worries\ab...  
3  Reuters - Authorities have halted oil export\f...  
4  AFP - Tearaway world oil prices, toppling reco...  


In [17]:
print(train_df.dtypes)

print(train_df.shape)
print(test_df.shape)

label     int64
title    object
text     object
dtype: object
(120000, 3)
(7600, 3)


In [18]:
train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

# Input and Output
X_train = train_df["text"].astype(str)
y_train = train_df["label"] - 1

X_test = test_df["text"].astype(str)
y_test = test_df["label"] - 1

print(X_train.head())
print(y_train.head())

0    Reuters - Short-sellers, Wall Street's dwindli...
1    Reuters - Private investment firm Carlyle Grou...
2    Reuters - Soaring crude prices plus worries\ab...
3    Reuters - Authorities have halted oil export\f...
4    AFP - Tearaway world oil prices, toppling reco...
Name: text, dtype: object
0    2
1    2
2    2
3    2
4    2
Name: label, dtype: int64


In [19]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

In [20]:
max_length = 100

X_train = pad_sequences(X_train, maxlen=max_length)

X_test = pad_sequences(X_test, maxlen=max_length)

In [21]:
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train, 4)
y_test = to_categorical(y_test, 4)

In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

model = Sequential()

model.add(Embedding(input_dim=10000,
                    output_dim=64,
                    input_length=100))

model.add(SimpleRNN(64))

model.add(Dense(32, activation="relu"))

model.add(Dense(4, activation="softmax"))

C:\Users\sandhya\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [23]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [24]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [25]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 112s 140ms/step - accuracy: 0.7856 - loss: 0.5494 - val_accuracy: 0.8619 - val_loss: 0.3904
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 94s 125ms/step - accuracy: 0.9077 - loss: 0.2815 - val_accuracy: 0.8818 - val_loss: 0.3475
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 108s 144ms/step - accuracy: 0.9319 - loss: 0.2071 - val_accuracy: 0.8727 - val_loss: 0.3814
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 99s 132ms/step - accuracy: 0.9544 - loss: 0.1371 - val_accuracy: 0.8655 - val_loss: 0.4518
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 95s 127ms/step - accuracy: 0.9719 - loss: 0.0866 - val_accuracy: 0.8693 - val_loss: 0.5335


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Loss :", loss)
print("Accuracy :", accuracy)

In [ ]:
sample = ["India won the cricket world cup"]

sequence = tokenizer.texts_to_sequences(sample)

sequence = pad_sequences(sequence, maxlen=100)

prediction = model.predict(sequence)

classes = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

print("Predicted Category :", classes[prediction.argmax()])